In [1]:
from sklearn.model_selection import cross_val_score, cross_val_predict, LeaveOneOut
from sklearn.metrics import confusion_matrix
from sklearn.svm import SVC, LinearSVC
cv = LeaveOneOut()

decoder = LinearSVC()

In [4]:
import numpy as np
import argparse
import logging
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io as sio
import seaborn as sns
import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC
from s2p_utils.data_loader import DataLoader
from s2p_utils.processing_utils import (
    correct_overlapping_cells_across_planes,
    correct_timestamps,
    get_cell_only_activity,
    extract_events,
    get_corrected_F,
    extract_interest_time_intervals,
    extract_imaging_ts_around_events,
    normalize_signal,
    extract_Fave_around_events,
    reorder_clusters,
)
from plot_utils import (
    plot_raw_licks,
    plot_average_PSTH_around_interest_window,
    plot_individual_cells_activity,
    plot_PC_screenplot,
    plot_PCs,
    make_silhouette_plot,
    plot_activity_clusters,
    plot_cluster_pairs,
    plot_individual_trial_average_activity,
)

In [5]:
trial_types = ["CS1+", "CS2+", "CS3-"]
number_trials = [25, 25, 50]
framerate = 5
delay_to_reward = 3
pre_cue_window = 3
post_cue_window = 17
learning_stage = 'early'

In [6]:
if learning_stage == "early":
    result_dir = "Z:\\2p\\experiment1\\population_data\\early learning\\"
    animal_list = [
        "MZ_CA1_WD_F3\\d1",
        "MZ_CA1_WD_M4\\d1",
        "MZ_CA1_WD_M5\\d1",
        "MZ_CA1_WD_M6\\d2",
        "MZ_CA1_WD_M7\\d1",
        "MZ_CA1_WD_M8\\d1",
        "MZ_CA1_WD_JB_54\\d3",
        "MZ_CA1_WD_JB_55\\d3"
    ]
elif learning_stage == "late":
    result_dir = "Z:\\2p\\experiment1\\population_data\\late learning\\"
    animal_list = [
        "MZ_CA1_WD_F3\\d7",
        "MZ_CA1_WD_M4\\d5",
        "MZ_CA1_WD_M5\\d6",
        "MZ_CA1_WD_M6\\d6",
        "MZ_CA1_WD_M7\\d5",
        "MZ_CA1_WD_M8\\d6",
        "MZ_CA1_WD_JB_54\\d8",
        "MZ_CA1_WD_JB_55\\d12"
    ]   
    

In [31]:
file_dir = 'Z:\\2p\\experiment1\\MZ_CA1_WD_F3\\d1\\files'
F_5hz = np.load(os.path.join(file_dir, "F_5hz.npy"), allow_pickle=True)
F = np.load(os.path.join(file_dir, "F.npy"), allow_pickle=True)

In [40]:
len(F_5hz[0][1])

17970

In [7]:
def extract_Fave_around_events(
    CS,
    F,
    im_ts,
    num_planes: int,
    pre_cue_window: int,
    post_cue_window: int,
):
    """
    This function first generates Fcorrected traces around each cues based on input images indexes,
    and average Fcorr across all CS trials within the same CS type for each cell,
    and append each cell's average activity under each cue.

    Args:
        CS: all CS trials
        F: Fcorrected trace for all planes all cells
        im_dx: image indexes around each cue
        num_planes: number of planes

    Returns:
    Fcorrected_around_cue with the structure of len(CS), number of cells, timepoints

    """
    
    # Extract time around each cue and sorted by CS type, shape is numCS --> len trials
    interest_intervals = extract_interest_time_intervals(
        CS, pre_cue_window, post_cue_window
    )
    # Extract image time points around each cue and sorted by CS type and plane, shape is plane --> numCS --> len trials
    im_idx_around_cue = extract_imaging_ts_around_events(
        CS, im_ts, num_planes, interest_intervals
    )

    # F_ave_around_cues = [[] for _ in range(len(CS))]
    F_ave_around_cues_baseline_subtract = [[] for _ in range(len(CS))]

    framenumber = len(
        F[0][0][im_idx_around_cue[0][0][1]]
    )  # reference frame number equals the first cell's second trial from the first plane
    framespersecond = framenumber // (pre_cue_window + post_cue_window)

    for cue_type, cs in enumerate(CS):  # cue_type = 0,1,2 (CS1, CS2, CS3)
        for ip in range(num_planes):
            cue_ts = im_idx_around_cue[ip][
                cue_type
            ]  # image indexes for all trials in this cue type, holds same for all cells within the plane (trial number x framenumber)
            for cell in range(len(F[ip])):
                cell_F = []
                for trial in range(len(cs)):
                    F_temp = F[ip][cell][
                        cue_ts[trial]
                    ]  # F for cell in the plane, of this trial in this cue type (framenumber x )
                    # Correct for frame for each trial
                    if len(F_temp) > framenumber:
                        # if images number is bigger than default, drop the extra ones
                        F_temp = F_temp[0:framenumber]
                    elif len(F_temp) < framenumber:
                        # if images is smaller than default, add nan at the end to fill the spots
                        for i in range(framenumber - len(F_temp)):
                            F_temp = np.append(F_temp, np.nan)
                    cell_F.append(F_temp)
                # average across cs trials
                # cellave = np.nanmean(np.array(cell_F), axis=0)
                # baseline = np.nanmean(cellave[0 : pre_cue_window * framespersecond])
                # baselinesubtract = list(cellave - baseline)
                F_ave_around_cues_baseline_subtract[cue_type].append(cell_F)
                # F_ave_around_cues[cue_type].append(cellave)
    F_ave_around_cues_baseline_subtract = np.array(F_ave_around_cues_baseline_subtract)
    return F_ave_around_cues_baseline_subtract
